# NB7 — LOO Error Analysis

Mục tiêu: giải thích **vì sao LOO Top-1 thấp hơn Hit@2** và xác định lỗi hiện tại là:

- **near-miss / ambiguity**: ground-truth đứng rank 2 nhưng delta rất sát Top-1;
- **clear ranking error**: ground-truth bị một item khác vượt xa;
- **length effect**: LOO yếu dần khi outfit dài hơn;
- **data-ground-truth ambiguity**: synthetic swapped item không nhất thiết là unique worst item theo scorer.

Notebook **không train lại model**. Nó phân tích các `loo_predictions_{valid,test}.jsonl` đã sinh từ NB6B.

Contract:

```text
Compatibility scorer: 2–8 items
LOO diagnosis: original outfit >= 3 items
Ground truth: negative_metadata.swapped_item_index
Ranking: delta_i = C(O \ x_i) - C(O), larger is more problematic
Tie break: lower item index, giống evaluate_loo.py
```


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from collections import Counter, defaultdict

EXPECTED_BRANCH = "exp/min2-scorer-loo3"
REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"

def is_project_repo(path: Path) -> bool:
    path = path.expanduser().resolve()
    return (
        (path / ".git").exists()
        and (path / "src/diagnosis/evaluate_loo.py").is_file()
        and (path / "configs/data_paths.min2_experiment.json").is_file()
    )

candidates = []
if os.environ.get("FASHION_PROJECT_ROOT"):
    candidates.append(Path(os.environ["FASHION_PROJECT_ROOT"]))

cwd = Path.cwd().resolve()
candidates.extend([cwd, *cwd.parents])
if Path("/content").exists():
    candidates.append(Path("/content/opisoverated"))
candidates.append(Path.home() / "opisoverated")

ROOT = next((p.expanduser().resolve() for p in candidates if is_project_repo(p)), None)

if ROOT is None:
    clone_parent = Path("/content") if Path("/content").exists() else Path.home()
    ROOT = clone_parent / "opisoverated"
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(
            f"{ROOT} exists but is not a usable checkout. "
            "Remove/rename it or set FASHION_PROJECT_ROOT."
        )
    subprocess.run(
        ["git", "clone", "--branch", EXPECTED_BRANCH, "--single-branch", REPO_URL, str(ROOT)],
        check=True,
    )

branch = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip()

if branch != EXPECTED_BRANCH:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", EXPECTED_BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", EXPECTED_BRANCH], check=True)

subprocess.run(
    ["git", "-C", str(ROOT), "pull", "--ff-only", "origin", EXPECTED_BRANCH],
    check=True,
)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("repo root :", ROOT)
print("branch    :", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip())
print("commit    :", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True
).strip())


## 1. Locate NB6B evaluation artifacts

NB6B lưu:

```text
evaluation_min2_exp_v1/
├── evaluation_summary.json
├── loo_predictions_valid.jsonl
└── loo_predictions_test.jsonl
```

Nếu NB7 chạy trong **cùng Colab runtime** với NB6B, notebook sẽ tìm thấy tự động.

Nếu bạn đã copy folder evaluation sang Google Drive, đặt `EVAL_DIR_OVERRIDE` thành path đó.

Ví dụ:

```python
EVAL_DIR_OVERRIDE = Path("/content/drive/MyDrive/ML_Final/min2/evaluation_min2_exp_v1")
```


In [ ]:
from src.data.runtime_paths import load_runtime_paths
from src.data.min2_experiment import scorer_ready_path

PATHS_CONFIG = ROOT / "configs/data_paths.min2_experiment.json"
paths = load_runtime_paths(repo_root=ROOT, config_path=PATHS_CONFIG)

EVAL_DIR_OVERRIDE = None  # Path("...") nếu evaluation artifacts nằm ở nơi khác.

default_eval_dir = paths.scorer_ready_dir / "evaluation_min2_exp_v1"
EVAL_DIR = Path(EVAL_DIR_OVERRIDE).expanduser().resolve() if EVAL_DIR_OVERRIDE else default_eval_dir

required_eval_files = [
    EVAL_DIR / "evaluation_summary.json",
    EVAL_DIR / "loo_predictions_valid.jsonl",
    EVAL_DIR / "loo_predictions_test.jsonl",
]

print("scorer-ready:", paths.scorer_ready_dir)
print("eval dir    :", EVAL_DIR)
for p in required_eval_files:
    print(p.name, "exists=", p.is_file())

if not all(p.is_file() for p in required_eval_files):
    raise FileNotFoundError(
        "Thiếu NB6B evaluation artifacts. Chạy NB6B tới cell Save evaluation artifacts "
        "trong cùng runtime, hoặc copy evaluation_min2_exp_v1 lên Drive và set EVAL_DIR_OVERRIDE."
    )


## 2. Load predictions + scorer-ready metadata

`evaluate_loo.py` đã lưu đầy đủ `loo_deltas`, `gt_delta`, `predicted_top1_delta`, `top2_indices`, v.v.

NB7 join thêm scorer-ready negative record để xem:

- item IDs trong outfit;
- `original_item_id`;
- `replacement_item_id`;
- `swap_category`;
- item mà model dự đoán problematic.


In [ ]:
import pandas as pd
import numpy as np

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

with (EVAL_DIR / "evaluation_summary.json").open("r", encoding="utf-8") as f:
    summary = json.load(f)

predictions = {
    split: read_jsonl(EVAL_DIR / f"loo_predictions_{split}.jsonl")
    for split in ("valid", "test")
}

scorer_rows = {
    split: read_jsonl(scorer_ready_path(paths.scorer_ready_dir, split))
    for split in ("valid", "test")
}
scorer_index = {
    split: {row["sample_id"]: row for row in rows if int(row.get("label", -1)) == 0}
    for split, rows in scorer_rows.items()
}

print("NB6B summary:")
print(json.dumps({
    "scorer_metrics": summary.get("scorer_metrics"),
    "loo_metrics": summary.get("loo_metrics"),
}, indent=2))

for split in ("valid", "test"):
    print(split, "LOO predictions=", len(predictions[split]),
          "negative scorer rows=", len(scorer_index[split]))


## 3. Derive per-sample error-analysis fields

### Ground-truth rank

Với mỗi sample:

\[
rank_{GT} = 1 + \#\{j: \Delta_j > \Delta_{GT}\}
\]

Notebook dùng **exact ranking rule giống evaluator**:

```python
sorted(indices, key=lambda i: (-delta[i], i))
```

### Margin

\[
margin = \Delta_{Top1} - \Delta_{GT}
\]

- `margin = 0`: GT chính là Top-1 hoặc exact tie được tie-break về GT;
- margin nhỏ: near-miss;
- margin lớn: scorer ưu tiên item khác rõ ràng hơn GT.


In [ ]:
def enrich_prediction(row, scorer_record):
    out = dict(row)
    deltas = [float(x) for x in row["loo_deltas"]]
    n = len(deltas)
    gt = int(row["gt_swapped_item_index"])
    ranked = sorted(range(n), key=lambda i: (-deltas[i], i))
    gt_rank = ranked.index(gt) + 1
    pred = ranked[0]

    meta = (scorer_record or {}).get("negative_metadata") or {}
    items = list((scorer_record or {}).get("items") or [])

    out.update({
        "gt_rank": gt_rank,
        "top1_margin_over_gt": float(deltas[pred] - deltas[gt]),
        "top1_minus_top2_margin": (
            float(deltas[ranked[0]] - deltas[ranked[1]]) if n >= 2 else np.nan
        ),
        "gt_item_id": items[gt] if gt < len(items) else meta.get("replacement_item_id"),
        "predicted_item_id": items[pred] if pred < len(items) else None,
        "original_swapped_out_item_id": meta.get("original_item_id"),
        "replacement_item_id": meta.get("replacement_item_id"),
        "swap_category": meta.get("swap_category"),
        "all_item_ids": items,
        "ranked_indices": ranked,
        "ranked_deltas": [deltas[i] for i in ranked],
    })
    return out

enriched = {}
frames = {}

for split in ("valid", "test"):
    rows = []
    missing_join = 0
    for row in predictions[split]:
        record = scorer_index[split].get(row["sample_id"])
        if record is None:
            missing_join += 1
        rows.append(enrich_prediction(row, record))
    enriched[split] = rows
    frames[split] = pd.DataFrame(rows)
    print(split, "rows=", len(rows), "missing scorer join=", missing_join)

assert all(len(df) > 0 for df in frames.values())


## 4. GT-rank distribution

Đây là câu trả lời trực tiếp cho gap **Top-1 vs Hit@2**.

Nếu phần lớn lỗi nằm ở `rank=2`, diagnosis đang **gần đúng**. Nếu nhiều case rơi vào rank 3+, scorer có vấn đề localization sâu hơn.


In [ ]:
def rank_distribution(df):
    counts = df["gt_rank"].value_counts().sort_index()
    total = len(df)
    rows = []
    for rank, count in counts.items():
        rows.append({
            "gt_rank": int(rank),
            "count": int(count),
            "fraction": float(count / total),
        })
    return pd.DataFrame(rows)

for split in ("valid", "test"):
    print("\n", split.upper())
    display(rank_distribution(frames[split]))


In [ ]:
import matplotlib.pyplot as plt

for split in ("valid", "test"):
    dist = rank_distribution(frames[split])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(dist["gt_rank"].astype(str), dist["fraction"])
    ax.set_xlabel("Ground-truth rank")
    ax.set_ylabel("Fraction of eligible negatives")
    ax.set_title(f"LOO GT-rank distribution — {split}")
    ax.set_ylim(0, 1)
    plt.show()


## 5. Random baseline + lift

Random Top-1 cho outfit dài `n` là `1/n`; random Hit@2 là `min(2/n, 1)`.

Ta tính baseline theo **exact length distribution** của mỗi split để tránh so LOO với 50% như binary classification.


In [ ]:
def random_baselines(df):
    top1 = float(np.mean(1.0 / df["outfit_length"].astype(float)))
    hit2 = float(np.mean(np.minimum(2.0 / df["outfit_length"].astype(float), 1.0)))
    observed_top1 = float(df["top1_correct"].mean())
    observed_hit2 = float(df["hit_at_2"].mean())
    return {
        "count": len(df),
        "random_top1": top1,
        "observed_top1": observed_top1,
        "top1_lift_x": observed_top1 / top1,
        "random_hit2": hit2,
        "observed_hit2": observed_hit2,
        "hit2_lift_x": observed_hit2 / hit2,
    }

baseline_rows = []
for split in ("valid", "test"):
    row = {"split": split, **random_baselines(frames[split])}
    baseline_rows.append(row)

display(pd.DataFrame(baseline_rows))


## 6. Rank-2 near-miss analysis

Tập quan trọng nhất:

```text
GT rank = 2
```

Ta đo:

```text
margin = delta_top1 - delta_GT
```

Thresholds mặc định:

- ≤ 0.01
- ≤ 0.05
- ≤ 0.10
- ≤ 0.25

Không coi các threshold này là metric chính thức; chúng chỉ giúp phân biệt **almost tie** với **clear preference for another item**.


In [ ]:
MARGIN_THRESHOLDS = [0.01, 0.05, 0.10, 0.25]

def margin_summary(df, rank=2):
    subset = df[df["gt_rank"] == rank].copy()
    margins = subset["top1_margin_over_gt"].astype(float)
    result = {
        "gt_rank": rank,
        "count": len(subset),
        "fraction_of_all": len(subset) / len(df),
        "mean_margin": float(margins.mean()) if len(subset) else np.nan,
        "median_margin": float(margins.median()) if len(subset) else np.nan,
        "p75_margin": float(margins.quantile(0.75)) if len(subset) else np.nan,
        "p90_margin": float(margins.quantile(0.90)) if len(subset) else np.nan,
    }
    for t in MARGIN_THRESHOLDS:
        result[f"margin_le_{t:.2f}"] = float((margins <= t).mean()) if len(subset) else np.nan
    return result

display(pd.DataFrame([
    {"split": split, **margin_summary(frames[split], rank=2)}
    for split in ("valid", "test")
]))


In [ ]:
for split in ("valid", "test"):
    rank2 = frames[split][frames[split]["gt_rank"] == 2]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(rank2["top1_margin_over_gt"], bins=40)
    ax.set_xlabel("delta_top1 - delta_GT")
    ax.set_ylabel("Count")
    ax.set_title(f"Rank-2 margin distribution — {split}")
    plt.show()


## 7. Error breakdown theo outfit length

Ta xem đồng thời:

- Top-1;
- Hit@2;
- GT rank trung bình;
- % GT rank=2;
- median margin của rank-2;
- mean GT delta;
- random Top-1 baseline.

Nếu length tăng mà `mean_gt_delta` giảm và rank xấu đi, đó là bằng chứng phù hợp với giả thuyết **LOO signal dilution** ở outfit dài.


In [ ]:
def by_length_analysis(df):
    rows = []
    for length, group in df.groupby("outfit_length", sort=True):
        rank2 = group[group["gt_rank"] == 2]
        rows.append({
            "outfit_length": int(length),
            "count": len(group),
            "top1": float(group["top1_correct"].mean()),
            "hit_at_2": float(group["hit_at_2"].mean()),
            "random_top1": 1.0 / float(length),
            "top1_lift_x": float(group["top1_correct"].mean()) / (1.0 / float(length)),
            "mean_gt_rank": float(group["gt_rank"].mean()),
            "rank2_fraction": float((group["gt_rank"] == 2).mean()),
            "rank3plus_fraction": float((group["gt_rank"] >= 3).mean()),
            "rank2_median_margin": (
                float(rank2["top1_margin_over_gt"].median()) if len(rank2) else np.nan
            ),
            "mean_gt_delta": float(group["gt_delta"].mean()),
            "mean_pred_top1_delta": float(group["predicted_top1_delta"].mean()),
        })
    return pd.DataFrame(rows)

length_tables = {}
for split in ("valid", "test"):
    length_tables[split] = by_length_analysis(frames[split])
    print("\n", split.upper())
    display(length_tables[split])


In [ ]:
for split in ("valid", "test"):
    table = length_tables[split]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(table["outfit_length"], table["top1"], marker="o", label="Observed Top-1")
    ax.plot(table["outfit_length"], table["random_top1"], marker="o", label="Random Top-1")
    ax.set_xlabel("Outfit length")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"LOO Top-1 vs random baseline — {split}")
    ax.legend()
    plt.show()

for split in ("valid", "test"):
    table = length_tables[split]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(table["outfit_length"], table["mean_gt_delta"], marker="o")
    ax.set_xlabel("Outfit length")
    ax.set_ylabel("Mean GT delta")
    ax.set_title(f"Ground-truth removal signal by outfit length — {split}")
    plt.show()


## 8. Category-level error analysis

Negative V1 swap giữ nguyên **master category**, nên `swap_category` cho biết loại item bị thay.

Bảng này tìm category nào LOO localization yếu hơn. Chỉ nên diễn giải category có đủ sample; mặc định lọc `count >= 20`.


In [ ]:
MIN_CATEGORY_COUNT = 20

def by_category_analysis(df, min_count=MIN_CATEGORY_COUNT):
    rows = []
    for category, group in df.groupby("swap_category", dropna=False):
        if len(group) < min_count:
            continue
        rank2 = group[group["gt_rank"] == 2]
        rows.append({
            "swap_category": category,
            "count": len(group),
            "top1": float(group["top1_correct"].mean()),
            "hit_at_2": float(group["hit_at_2"].mean()),
            "mean_gt_rank": float(group["gt_rank"].mean()),
            "rank2_fraction": float((group["gt_rank"] == 2).mean()),
            "rank2_median_margin": (
                float(rank2["top1_margin_over_gt"].median()) if len(rank2) else np.nan
            ),
            "mean_gt_delta": float(group["gt_delta"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["top1", "count"], ascending=[True, False])

category_tables = {}
for split in ("valid", "test"):
    category_tables[split] = by_category_analysis(frames[split])
    print("\nWeakest categories —", split)
    display(category_tables[split].head(15))


## 9. Inspect concrete rank-2 cases

Hai nhóm hữu ích:

1. **Ambiguous rank-2:** margin nhỏ nhất — GT gần như ngang Top-1.
2. **Clear rank-2 error:** margin lớn nhất — scorer thật sự thích loại item khác hơn GT.

Cột `all_item_ids` + `ranked_indices` cho phép quay lại ảnh/item metadata nếu cần visual inspection.


In [ ]:
CASE_COLUMNS = [
    "sample_id",
    "outfit_length",
    "swap_category",
    "gt_swapped_item_index",
    "predicted_problematic_index",
    "gt_item_id",
    "predicted_item_id",
    "original_swapped_out_item_id",
    "replacement_item_id",
    "gt_delta",
    "predicted_top1_delta",
    "top1_margin_over_gt",
    "original_logit",
    "all_item_ids",
    "ranked_indices",
    "ranked_deltas",
]

N_CASES = 20

for split in ("valid", "test"):
    rank2 = frames[split][frames[split]["gt_rank"] == 2].copy()

    print("\n", "=" * 80)
    print(split.upper(), "— most ambiguous rank-2 cases")
    display(
        rank2.sort_values("top1_margin_over_gt", ascending=True)[CASE_COLUMNS].head(N_CASES)
    )

    print(split.upper(), "— clearest rank-2 errors")
    display(
        rank2.sort_values("top1_margin_over_gt", ascending=False)[CASE_COLUMNS].head(N_CASES)
    )


## 10. Inspect rank 3+ failures

Nếu nhiều lỗi rank 3+, đây không còn là “Top-1 vs Top-2 ambiguity”.

Ta ưu tiên các case có:

- GT rank cao;
- margin lớn;
- outfit length nhỏ (đặc biệt n=3 hoặc n=4), vì đây là failure khó giải thích bằng candidate count đơn thuần.


In [ ]:
for split in ("valid", "test"):
    hard = frames[split][frames[split]["gt_rank"] >= 3].copy()
    hard = hard.sort_values(
        ["gt_rank", "top1_margin_over_gt", "outfit_length"],
        ascending=[False, False, True],
    )
    print("\nHard failures —", split, "count=", len(hard))
    display(hard[CASE_COLUMNS + ["gt_rank"]].head(30))


## 11. Decision-oriented summary

Notebook tạo 4 số cần nhìn trước khi quyết định sửa architecture:

1. `% rank=1` → exact localization;
2. `% rank=2` → recoverable near-miss pool;
3. `% rank>=3` → deeper localization failure;
4. trong rank=2, `% margin <= 0.05 / 0.10` → mức ambiguity.

Heuristic cho bước tiếp theo:

```text
rank=2 nhiều + margin nhỏ
    -> LOO formulation đang gần đúng; cân nhắc confidence / Top-2 UI / tie-aware diagnosis.

rank=2 nhiều + margin lớn
    -> scorer không align với swapped-item localization;
       diagnosis-aware ranking loss đáng thử.

rank>=3 nhiều, nhất là n=3/4
    -> cần xem lại scorer representation / negative protocol trước khi thêm loss.

chỉ xấu ở n>=5 và mean_gt_delta giảm mạnh
    -> investigate pairwise mean aggregation / signal dilution theo outfit length.
```

Đây là **diagnostic heuristic**, không phải acceptance criterion chính thức.


In [ ]:
def decision_summary(df, split):
    rank1 = float((df["gt_rank"] == 1).mean())
    rank2 = float((df["gt_rank"] == 2).mean())
    rank3plus = float((df["gt_rank"] >= 3).mean())
    rank2_df = df[df["gt_rank"] == 2]
    return {
        "split": split,
        "eligible_count": len(df),
        "rank1_fraction": rank1,
        "rank2_fraction": rank2,
        "rank3plus_fraction": rank3plus,
        "rank2_margin_median": (
            float(rank2_df["top1_margin_over_gt"].median()) if len(rank2_df) else np.nan
        ),
        "rank2_margin_le_0.05": (
            float((rank2_df["top1_margin_over_gt"] <= 0.05).mean())
            if len(rank2_df) else np.nan
        ),
        "rank2_margin_le_0.10": (
            float((rank2_df["top1_margin_over_gt"] <= 0.10).mean())
            if len(rank2_df) else np.nan
        ),
        "mean_gt_delta": float(df["gt_delta"].mean()),
        "mean_pred_top1_delta": float(df["predicted_top1_delta"].mean()),
    }

decision_df = pd.DataFrame([
    decision_summary(frames[split], split)
    for split in ("valid", "test")
])
display(decision_df)


## 12. Save NB7 artifacts

Output:

```text
evaluation_min2_exp_v1/loo_error_analysis/
├── loo_error_analysis_summary.json
├── per_sample_valid.csv
├── per_sample_test.csv
├── by_length_valid.csv
├── by_length_test.csv
├── by_category_valid.csv
├── by_category_test.csv
├── rank2_ambiguous_test.csv
├── rank2_clear_errors_test.csv
└── rank3plus_hard_failures_test.csv
```

Không overwrite NB6B predictions.


In [ ]:
ANALYSIS_DIR = EVAL_DIR / "loo_error_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

analysis_summary = {
    "source_evaluation_dir": str(EVAL_DIR),
    "source_checkpoint": summary.get("checkpoint"),
    "dataset_version": summary.get("dataset_version"),
    "scorer_min_items": summary.get("scorer_min_items"),
    "loo_min_original_items": summary.get("loo_min_original_items"),
    "decision_summary": {
        row["split"]: {k: v for k, v in row.items() if k != "split"}
        for row in decision_df.to_dict(orient="records")
    },
    "random_baselines": {
        row["split"]: {k: v for k, v in row.items() if k != "split"}
        for row in baseline_rows
    },
    "margin_thresholds": MARGIN_THRESHOLDS,
}

with (ANALYSIS_DIR / "loo_error_analysis_summary.json").open("w", encoding="utf-8") as f:
    json.dump(analysis_summary, f, ensure_ascii=False, indent=2)
    f.write("\n")

for split in ("valid", "test"):
    frames[split].to_csv(ANALYSIS_DIR / f"per_sample_{split}.csv", index=False)
    length_tables[split].to_csv(ANALYSIS_DIR / f"by_length_{split}.csv", index=False)
    category_tables[split].to_csv(ANALYSIS_DIR / f"by_category_{split}.csv", index=False)

test_rank2 = frames["test"][frames["test"]["gt_rank"] == 2].copy()
test_rank2.sort_values("top1_margin_over_gt").head(100).to_csv(
    ANALYSIS_DIR / "rank2_ambiguous_test.csv", index=False
)
test_rank2.sort_values("top1_margin_over_gt", ascending=False).head(100).to_csv(
    ANALYSIS_DIR / "rank2_clear_errors_test.csv", index=False
)
frames["test"][frames["test"]["gt_rank"] >= 3].sort_values(
    ["gt_rank", "top1_margin_over_gt"], ascending=[False, False]
).head(100).to_csv(
    ANALYSIS_DIR / "rank3plus_hard_failures_test.csv", index=False
)

print(json.dumps(analysis_summary, indent=2))
print("saved to:", ANALYSIS_DIR)


## Acceptance / interpretation checklist

Sau khi chạy NB7, trả lời lần lượt:

1. Test error có chủ yếu là `GT rank=2` không?
2. Trong `GT rank=2`, bao nhiêu % có margin ≤ 0.05 hoặc ≤ 0.10?
3. `rank>=3` có tập trung ở outfit dài không?
4. Length=3 có nhiều rank>=3 không? Nếu có, đây là red flag cho MIN2/LOO boundary.
5. Category nào consistently yếu ở cả valid và test?
6. Các clear rank-2 errors có hợp lý về mặt thời trang khi nhìn item thực tế không?
7. Nếu GT bị model vượt xa nhưng predicted item nhìn cũng thực sự lạc outfit, cần cân nhắc rằng `swapped_item_index` là synthetic provenance, không nhất thiết là unique semantic outlier.

**Không thay architecture trước khi trả lời được các câu trên.**
